# NewsLens AI · 02 · Fake-news model development
This notebook inspects the saved, measured training outputs. The authoritative reproducible implementation is `training/train_fake_news_models.py`; using one script prevents notebook-state leakage.

In [ ]:
from pathlib import Path
import json, joblib, pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
comparison = pd.read_csv(ROOT / 'reports/results/model_comparison.csv')
metrics = json.loads((ROOT / 'reports/results/model_metrics.json').read_text())
display(comparison.sort_values('macro_f1', ascending=False))
metrics

## Leakage-safe pipeline
`GridSearchCV` receives a complete Pipeline: raw cleaned text → training-fold TF-IDF → classifier. Therefore vocabulary and inverse-document-frequency weights are never fitted on the held-out set. Selection prioritises macro-F1 and interpretable probability output rather than accuracy alone.

In [ ]:
pipeline = joblib.load(ROOT / 'models/fake_news_pipeline.joblib')
pipeline.named_steps, pipeline.get_params()['classifier']

In [ ]:
from src.fake_news_predictor import predict_credibility
sample = (ROOT / 'data/sample/uncertain_style_article.txt').read_text()
result = predict_credibility(sample, pipeline)
result.to_dict()

## Rebuild
From the project terminal run `python training/download_data.py --dataset isot` and then `python training/train_fake_news_models.py`. Expect results to remain deterministic for the recorded dependency releases and dataset hashes, subject to platform-level floating-point differences.